# 00 — Data Preparation

**Goal:** produce `data/processed/pancreas_prepped.h5ad`, the single canonical
object every other notebook consumes. Nothing downstream is correct if this isn't.

**Output schema** (asserted at the end of this notebook):

| Field | Contents |
|---|---|
| `X` | log-normalized expression, HVG-subset |
| `layers['counts']` | raw integer counts (scVI + the from-scratch VAE need these) |
| `obs['batch']` | sequencing technology, categorical |
| `obs['cell_type']` | curated annotation, categorical |
| `var['ensembl_id']` | Ensembl gene IDs (Geneformer tokenizer) |
| `obsm['X_pca']` | uncorrected PCA — the control arm + the "before" UMAP |

Cells 3–7 are prototyped here, then lifted into `src/data.py::load_pancreas()` once stable.

In [2]:
# Cell 2 — Setup: make src importable, seed, point scanpy at project dirs.

import sys, os
# Put the repo root on sys.path so `from src...` resolves regardless of kernel CWD.
# Assumes the kernel's working dir is notebooks/ (VS Code's default) → repo root is "..".
sys.path.insert(0, os.path.abspath(".."))

import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd

from src.utils import set_seeds, DATA_RAW, DATA_PROCESSED, FIGURES
set_seeds(0)

# Point scanpy at the project dirs (mirrors your notebook 01)
sc.settings.datasetdir = str(DATA_RAW)   # scanpy's downloaded datasets land here
sc.settings.figdir     = str(FIGURES)

%load_ext autoreload
%autoreload 2

print("Processed data will be saved to:", DATA_PROCESSED)

d:\Anaconda\envs\comp_bio\Lib\site-packages\louvain\__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


Processed data will be saved to: E:\Data_Science_Bio\multi-batch-integration\data\processed


In [3]:
# Cell 3 — Load the scIB pancreas benchmark, then interrogate it.
#
# The loader is filled in (verified source). The interrogation is yours: the
# point is to *look* and decide, because every later cell trusts what you conclude.

# ~126 MB download to data/raw on first run; reads locally on every run after.
adata = sc.read(
    DATA_RAW / "pancreas.h5ad",
    backup_url="https://figshare.com/ndownloader/files/24539828",
)
adata   # expect: 16382 x 19093, obs ['tech','celltype','size_factors'], layers ['counts']

# --- Interrogate. Answer three questions before moving on. ---
#
# Q1. BATCH + LABELS: which obs column is the batch, which is the annotation?
#     TODO: value_counts() on 'tech' (how many technologies?) and on 'celltype'
#     (how many cell types?).
#
# Q2. WHAT IS IN X?  The one that matters. counts is in a layer, so X is some
#     processed form — but which?
#     TODO: compare X against layers['counts']. Check X for negative values
#     (negatives => scaled/z-scored), integer vs float, and the min/max of each.
#     Decide: is X log-normalized, scaled, or something else? You're going to
#     recompute it from counts regardless — this just tells you what you're replacing.
#
# Q3. GENE IDS: are var_names gene symbols or Ensembl IDs?
#     TODO: look at adata.var and adata.var_names. Geneformer needs Ensembl IDs,
#     so this tells you how much mapping notebook 00 still owes.

AnnData object with n_obs × n_vars = 16382 × 19093
    obs: 'tech', 'celltype', 'size_factors'
    layers: 'counts'

In [8]:
adata.obs

,tech,celltype,size_factors
D101_5,celseq,gamma,0.028492
D101_43,celseq,gamma,0.079348
D101_93,celseq,gamma,0.037932
D102_4,celseq,gamma,0.047685
D172444_23,celseq,gamma,0.038683
...,...,...,...
Sample_1594,smarter,gamma,1.000000
Sample_1595,smarter,gamma,1.000000
Sample_1597,smarter,gamma,1.000000
Sample_1598,smarter,gamma,1.000000


In [4]:
# Q1. BATCH + LABELS: which obs column is the batch, which is the annotation?
#     TODO: value_counts() on 'tech' (how many technologies?) and on 'celltype'
#     (how many cell types?).

adata.obs.tech.value_counts() # how many tech

tech
inDrop3       3605
smartseq2     2394
celseq2       2285
inDrop1       1937
inDrop2       1724
smarter       1492
inDrop4       1303
celseq        1004
fluidigmc1     638
Name: count, dtype: int64

In [5]:
adata.obs.celltype.value_counts() # how many cell types

celltype
alpha                 5493
beta                  4169
ductal                2142
acinar                1669
delta                 1055
gamma                  699
activated_stellate     464
endothelial            313
quiescent_stellate     193
macrophage              79
mast                    42
epsilon                 32
schwann                 25
t_cell                   7
Name: count, dtype: int64

In [6]:
pd.crosstab(adata.obs.tech, adata.obs.celltype) # tech vs # celltype

celltype,acinar,activated_stellate,alpha,beta,delta,ductal,endothelial,epsilon,gamma,macrophage,mast,quiescent_stellate,schwann,t_cell
tech,,,,,,,,,,,,,,
celseq,228,19,191,161,50,327,5,1,18,1,1,1,1,0
celseq2,274,90,843,445,203,258,21,4,110,15,6,12,4,0
fluidigmc1,21,16,239,258,25,36,14,1,18,1,3,1,5,0
inDrop1,110,51,236,872,214,120,130,13,70,14,8,92,5,2
inDrop2,3,81,676,371,125,301,23,2,86,17,9,22,6,2
inDrop3,843,100,1130,787,161,376,92,2,36,14,7,54,1,2
inDrop4,2,52,284,495,101,280,7,1,63,10,1,5,1,1
smarter,0,0,886,472,49,0,0,0,85,0,0,0,0,0
smartseq2,188,55,1008,308,127,444,21,8,213,7,7,6,2,0


In [33]:
# Q2. WHAT IS IN X?  The one that matters. counts is in a layer, so X is some
#     processed form — but which?
#     TODO: compare X against layers['counts']. Check X for negative values
#     (negatives => scaled/z-scored), integer vs float, and the min/max of each.
#     Decide: is X log-normalized, scaled, or something else? You're going to
#     recompute it from counts regardless — this just tells you what you're replacing.

import numpy as np
from scipy.sparse import issparse

# First 50 cells as a dense array (works whether X is sparse or dense)
X_head = adata.X[:50]
X_head = X_head.toarray() if issparse(X_head) else np.asarray(X_head)
C_head = adata.layers['counts'][:50]
C_head = C_head.toarray() if issparse(C_head) else np.asarray(C_head)

# 1. Negatives? (scaled / z-scored data has them; counts and log-norm don't)
print("X   min / max:", adata.X.min(), adata.X.max())

# 2. Whole numbers (raw counts) or fractional (transformed)?
print("X first row, 15 vals:", np.round(X_head[0, :15], 3))
print("X integer-valued? ", np.allclose(X_head, np.round(X_head)))

# 3. Confirm the counts layer really is raw
print("counts min / max:", adata.layers['counts'].min(), adata.layers['counts'].max())
print("counts integer-valued?", np.allclose(C_head, np.round(C_head)))

# 4. Is X just a copy of counts, or something processed?
print("X identical to counts (first 50 cells)?", np.array_equal(X_head, C_head))

X   min / max: 0.0 13.002677
X first row, 15 vals: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
X integer-valued?  False
counts min / max: 0.0 1.453667e+06
counts integer-valued? False
X identical to counts (first 50 cells)? False


In [ ]:
# Based on observations, counts[layers] consists of the raw values while X consists of 
# log1p(normalize(counts)) values, i.e. a transformed version of those counts: normalized 
# for library size (each cell scaled so total counts are comparable across cells) and 
# then log1p-transformed — i.e., log-normalized expression, non-negative, fractional values.

In [32]:
# Q3. GENE IDS: are var_names gene symbols or Ensembl IDs?
#     TODO: look at adata.var and adata.var_names. Geneformer needs Ensembl IDs,
#     so this tells you how much mapping notebook 00 still owes.

# adata.var
adata.var_names

Index(['A1BG', 'A1CF', 'A2M', 'A2ML1', 'A4GALT', 'A4GNT', 'AA06', 'AAAS',
       'AACS', 'AACSP1',
       ...
       'ZW10', 'ZWILCH', 'ZWINT', 'ZXDA', 'ZXDB', 'ZXDC', 'ZYG11B', 'ZYX',
       'ZZEF1', 'ZZZ3'],
      dtype='object', length=19093)

In [34]:
from scipy.sparse import issparse

def densify(m):
    return m.toarray() if issparse(m) else np.asarray(m)

print(f"{'protocol':<12}{'counts integer?':<18}{'max':>14}")
for tech in adata.obs['tech'].unique():
    mask = (adata.obs['tech'] == tech).values
    sub = densify(adata.layers['counts'][mask][:50])
    print(f"{tech:<12}{str(np.allclose(sub, np.round(sub))):<18}{sub.max():>14.1f}")

protocol    counts integer?              max
celseq      False                     1597.0
celseq2     False                     1597.0
fluidigmc1  False                   520845.0
smartseq2   True                    367998.0
inDrop1     True                      3375.0
inDrop2     True                      3476.0
inDrop3     True                      3040.0
inDrop4     True                      4234.0
smarter     False                   113248.7


In [ ]:
# Cell 4 — Standardize to the canonical schema.
# X stays as-is (log-normalized, confirmed in Q2). counts layer stays untouched.
# Only job here: give the two columns the names every downstream cell expects,
# as ordered categoricals.

# 1. Rename obs columns:  'tech' -> 'batch',  'celltype' -> 'cell_type'
#    TODO: adata.obs.rename(columns={...}, inplace=True)

# 2. Cast both to categorical dtype (scanpy/scvi expect this; avoids silent
#    string-vs-category bugs later).
#    TODO: adata.obs['batch']     = adata.obs['batch'].astype('category')
#          adata.obs['cell_type'] = adata.obs['cell_type'].astype('category')

# 3. Verify: print adata.obs.columns, and confirm 'batch'/'cell_type' exist,
#    are categorical, and still sum to 16382 with the same category counts.